In [1]:
# Import the libraries
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import time
from datetime import datetime as dt
import numpy as np
import pickle
import matplotlib.dates as mdates
import spacepy.pycdf as pycdf
import os

/Users/quocanh1306/Desktop/miniconda3/envs/qanh-env/lib/python3.10/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


# Load the monthly in-situ data

In [2]:
# Read the insitu file
dir_insitu = '/Users/quocanh1306/Downloads/Insitu_05_2016.csv'
df_insitu = pd.read_csv(dir_insitu)
df_insitu.head()

# Convert datetime
df_insitu["datetime"] = pd.to_datetime(df_insitu["datetime"])
#print(df_insitu.info())

In [3]:
print(df_insitu)

                  datetime  altitude     SZA      B_x   B_y     B_z    GLON  \
0      2016-05-01 00:00:00   2813.68   91.44 -0.84500 -3.95  0.0452  260.01   
1      2016-05-01 00:00:08   2803.74   91.33 -1.66000 -3.14  2.2500  260.06   
2      2016-05-01 00:00:16   2793.78   91.22 -0.62200 -4.50  0.1680  260.11   
3      2016-05-01 00:00:24   2783.81   91.11 -0.36800 -5.34  0.2920  260.17   
4      2016-05-01 00:00:32   2773.83   90.99 -0.00941 -3.57  2.0800  260.22   
...                    ...       ...     ...      ...   ...     ...     ...   
363484 2016-05-31 23:59:20   4298.67  112.86 -2.26000 -2.90 -0.9670   78.92   
363485 2016-05-31 23:59:28   4291.18  112.89 -5.51000 -1.85 -1.8400   79.31   
363486 2016-05-31 23:59:36   4283.68  112.91 -3.67000  2.92  3.0500   79.70   
363487 2016-05-31 23:59:44   4276.15  112.93  0.05850 -2.73  7.4600   80.09   
363488 2016-05-31 23:59:52   4268.62  112.96      NaN   NaN     NaN   80.48   

         GLAT  flux_parallel  flux_antiparallel  
0

# Load the monthly PAD data

In [4]:
# Create empty lists to store data
b_azim = []
b_elev = []
binning = []
counts = []
d_pa = []
de_over_e = []
diff_en_fluxes = []
en_label = []
energy = []
epoch = []
g_engy = []
g_pa = []
geom_factor = []
num_dists = []
pa = []
pa_label = []
pindex = []
time_met = []
time_unix = []

In [5]:
# Path to saved PAD data
path_in = '/Volumes/DATA/Research Project/Mars/Data/PAD'

# List of years and months data
years = ['2016']
months = ['05']

In [6]:
# Run this loop to read the files and necessary columns
for year in years:
 path_year = path_in + f'/{year}'
 for month in months:
    path = path_year + f'/{month}/'
    for filename in os.listdir(path):
      if filename.endswith('.cdf'):
       print(filename)
       filepath = os.path.join(path, filename)
       readfile = pycdf.CDF(filepath)
       b_azim.append(readfile['b_azim'][:])
       b_elev.append(readfile['b_elev'][:])
       diff_en_fluxes.append(readfile['diff_en_fluxes'][:])
       pa.append(readfile['pa'][:])
       energy.append(readfile['energy'][:])
       epoch.append(readfile['epoch'][:])

mvn_swe_l2_svypad_20160501_v05_r01.cdf
mvn_swe_l2_svypad_20160502_v05_r01.cdf


In [7]:
b_azim_all = np.concatenate(b_azim)
b_elev_all = np.concatenate(b_elev)
energ_all = energy[0]
epoch_all = np.concatenate(epoch)
diff_en_fluxes_all = np.concatenate(diff_en_fluxes)
pa_all = np.concatenate(pa)

In [8]:
# Convert other columns into numpy array type
b_azim_1m = np.squeeze(np.array(b_azim_all))
b_elev_1m = np.squeeze(np.array(b_elev_all))
energy_1m = np.squeeze(np.array(energ_all))
epoch_1m = np.squeeze(np.array(epoch_all))



In [9]:
import gc # Garbage Collector

# ... your existing loop code ...

# --- MODIFIED SECTION ---
# Create the numpy array
diff_en_fluxes_1m = np.squeeze(np.array(diff_en_fluxes_all))

# CRITICAL STEP: Delete the list immediately to free 11GB!
del diff_en_fluxes_all
gc.collect() 

# Repeat for other large variables
pa_1m = np.squeeze(np.array(pa_all))
del pa_all
gc.collect()

0

In [9]:
print(len(b_azim_1m))
print(len(b_elev_1m ))
print(len(energy_1m))
print(len(epoch_1m))
print(len(diff_en_fluxes_1m))
print(len(pa_1m))

1294456
1294456
64
1294456
1294456
1294456


In [10]:
print(b_azim_1m.shape)
print(b_elev_1m.shape)
print(energy_1m.shape)
print(epoch_1m.shape)
print(diff_en_fluxes_1m.shape)
print(pa_1m.shape)

(1294456,)
(1294456,)
(64,)
(1294456,)
(1294456, 16, 64)
(1294456, 16, 64)


In [10]:
import xarray as xr

In [11]:

# 1. Verify shapes (based on your screenshot)
# diff_en_fluxes_1m: (734727, 16, 64)
# pa_1m:             (734727, 16, 64)
# b_azim_1m:         (734727,)
# energy_1m:         (64,)

# 2. Create an Xarray Dataset
# We map dimensions to the arrays. 
# Note how we handle 1D arrays (b_azim) and 3D arrays (flux) together efficiently.

ds_0 = xr.Dataset(
    data_vars={
        # 3D Variables: Depend on time, pitch angle bin, and energy
        "def": (("time", "pa_bin", "energy"), diff_en_fluxes_1m),
        "pa values": (("time", "pa_bin", "energy"), pa_1m),
        
        # 1D Variables: Depend ONLY on time (Xarray handles the broadcasting for you!)
        "b_azim": (("time"), b_azim_1m),
        "b_elev": (("time"), b_elev_1m),
    },
    coords={
        "time": epoch_1m,
        "energy": energy_1m,
        "pa_bin": np.arange(16),  # Simple index 0-15
    }
)

# 3. View the result (Instantaneous, no memory explosion)
df_pad = ds_0
print(df_pad)

<xarray.Dataset> Size: 697MB
Dimensions:    (time: 84872, pa_bin: 16, energy: 64)
Coordinates:
  * time       (time) datetime64[ns] 679kB 2016-05-01T00:00:01.940054 ... 201...
  * energy     (energy) float32 256B 4.628e+03 4.119e+03 3.665e+03 ... 3.371 3.0
  * pa_bin     (pa_bin) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Data variables:
    def        (time, pa_bin, energy) float32 348MB 0.0 0.0 ... 7.456e+07
    pa values  (time, pa_bin, energy) float32 348MB 12.04 12.04 ... 28.3 28.3
    b_azim     (time) float32 339kB 28.83 336.8 327.0 ... 124.5 130.1 127.3
    b_elev     (time) float32 339kB -6.75 6.75 -2.25 11.25 ... 6.75 -2.25 2.25


In [12]:
import gc

# Delete the input variables to free up their space
del diff_en_fluxes_1m
del pa_1m
del b_azim_1m
del b_elev_1m

# Force Python to release the memory
gc.collect()

40

In [13]:
df_pad_clean_2 = df_pad.drop_duplicates(dim='time', keep='first')

# Safety Check: Verify it is actually unique now
is_unique = df_pad_clean_2.indexes['time'].is_unique
#print(f"Dataset time index is unique: {is_unique}")
#print(f"Original size: {df_pad.sizes['time']}")
#print(f"Cleaned size:  {df_pad_clean_2.sizes['time']}")

if not is_unique:
    raise ValueError("Critical Error: The dataset still has duplicate times even after cleaning!")

In [14]:
# --- STEP 1: FORCE CLEAN MAVEN DATA (df_pad_clean_2) ---
# We apply drop_duplicates again just to be 100% safe
df_maven_final = df_pad_clean_2.drop_duplicates(dim='time', keep='first')
maven_unique_times = df_maven_final.time.to_index().drop_duplicates()


# Verify immediately
if not df_maven_final.indexes['time'].is_unique:
    raise ValueError("CRITICAL: MAVEN data still has duplicates! Check your loading step.")

print(f"MAVEN Data Clean. Time points: {df_maven_final.sizes['time']}")

# --- STEP 2: FORCE CLEAN IN-SITU DATA ---
# 1. Sort
df_insitu_sorted = df_insitu.sort_values('datetime')
# 2. Drop duplicates (Pandas method)
df_insitu_unique = df_insitu_sorted.drop_duplicates(subset='datetime', keep='first')
# 3. Convert to Xarray and Rename
ds_insitu_clean = df_insitu_unique.set_index('datetime').to_xarray().rename({'datetime': 'time'})

print(f"In-Situ Data Clean. Time points: {ds_insitu_clean.sizes['time']}")

# --- STEP 3: ALIGNMENT ---
# Select in-situ data matching the MAVEN timeline
# Since df_maven_final is guaranteed unique now, this result will also be unique.
aligned_insitu = ds_insitu_clean.reindex(
    time=maven_unique_times, 
    method='nearest', 
    tolerance=np.timedelta64(4, 's')
)

# --- STEP 4: MERGE ---
# Now both datasets share the exact same 'time' index (maven_unique_times).
# The merge will simply combine columns side-by-side.
df_final = xr.merge([df_maven_final, aligned_insitu])
# Verify: The size should strictly match MAVEN's size
#print(f"Final Size: {df_final.sizes['time']}")
#print(df_final)

MAVEN Data Clean. Time points: 84872
In-Situ Data Clean. Time points: 363489


In [15]:
# Anti-Parallel: 120 <= PA <= 180
cond_antiparallel = (df_final['pa values'] >= 120) & (df_final['pa values'] <= 180)

# Parallel: 0 <= PA <= 60 (Assuming 60 to match the width of the anti-parallel sector)
cond_parallel = (df_final['pa values'] >= 0) & (df_final['pa values'] <= 60)

flux_anti_3d = df_final['def'].where(cond_antiparallel)

flux_para_3d = df_final['def'].where(cond_parallel)

In [16]:
# Result shape: (Time, Energy)
avg_flux_anti = flux_anti_3d.mean(dim='pa_bin', skipna=True)

avg_flux_para = flux_para_3d.mean(dim='pa_bin', skipna=True)

In [17]:
df_final['calc_flux_antiparallel'] = avg_flux_anti
df_final['calc_flux_parallel'] = avg_flux_para

print(df_final)

<xarray.Dataset> Size: 746MB
Dimensions:                 (time: 84872, pa_bin: 16, energy: 64)
Coordinates:
  * time                    (time) datetime64[ns] 679kB 2016-05-01T00:00:01.9...
  * energy                  (energy) float32 256B 4.628e+03 4.119e+03 ... 3.0
  * pa_bin                  (pa_bin) int64 128B 0 1 2 3 4 5 ... 11 12 13 14 15
Data variables: (12/15)
    def                     (time, pa_bin, energy) float32 348MB 0.0 ... 7.45...
    pa values               (time, pa_bin, energy) float32 348MB 12.04 ... 28.3
    b_azim                  (time) float32 339kB 28.83 336.8 ... 130.1 127.3
    b_elev                  (time) float32 339kB -6.75 6.75 -2.25 ... -2.25 2.25
    altitude                (time) float64 679kB 2.814e+03 ... 6.16e+03
    SZA                     (time) float64 679kB 91.44 91.44 ... 124.3 124.3
    ...                      ...
    GLON                    (time) float64 679kB 260.0 260.0 ... 151.3 151.3
    GLAT                    (time) float64 679kB -41

In [18]:
# Delete the standalone variables to free space
del avg_flux_anti
del avg_flux_para

# Force the garbage collector to release the memory immediately
gc.collect()

0

In [19]:
df_final = df_final.chunk({'time': 10000})

In [20]:
# The function to calculate the flux derivative using Lagrangian 3=point 
def calculate_slope_vectorized(ds, flux_var_name):
    """
    Applies the 3-point Lagrangian derivative to the entire dataset at once.
    Math is identical to your 'lagrangian_derivative' function.
    """
    # 1. Prepare X (Energy) and Y (Flux) in Log10 space
    # We use 'where' to avoid log(0) errors if flux is 0 or negative
    x_vals = np.log10(ds['energy'])
    f_vals = np.log10(ds[flux_var_name].where(ds[flux_var_name] > 0))
    
    # 2. Get the shifted arrays for x1, x2, x3 and f1, f2, f3
    # Shift(1)  -> moves data right (gets the 'previous' index, i-1)
    # Shift(-1) -> moves data left  (gets the 'next' index, i+1)
    
    # Points at index - 1 (x1, f1)
    x1 = x_vals.shift(energy=1)
    f1 = f_vals.shift(energy=1)
    
    # Points at index (x2, f2) - The center point
    x2 = x_vals
    f2 = f_vals
    
    # Points at index + 1 (x3, f3)
    x3 = x_vals.shift(energy=-1)
    f3 = f_vals.shift(energy=-1)
    
    # 3. Apply your Formula
    # term1 = f1 * (2*x2 - x3 - x2) / ((x1 - x2) * (x1 - x3))
    term1 = f1 * (x2 - x3) / ((x1 - x2) * (x1 - x3)) # Simplified (2*x2 - x2 = x2)
    
    # term2 = f2 * (2*x2 - x1 - x3) / ((x2 - x1) * (x2 - x3))
    term2 = f2 * (2 * x2 - x1 - x3) / ((x2 - x1) * (x2 - x3))
    
    # term3 = f3 * (2*x2 - x1 - x2) / ((x3 - x1) * (x3 - x2))
    term3 = f3 * (x2 - x1) / ((x3 - x1) * (x3 - x2)) # Simplified (2*x2 - x2 = x2)
    
    derivative = (term1 + term2 + term3)*(-0.05)
    
    return derivative

In [21]:
# 1. Calculate for Anti-Parallel
df_final['slope_antiparallel'] = calculate_slope_vectorized(df_final, 'calc_flux_antiparallel')

# 2. Calculate for Parallel
df_final['slope_parallel'] = calculate_slope_vectorized(df_final, 'calc_flux_parallel')

In [22]:
print(df_final)

<xarray.Dataset> Size: 790MB
Dimensions:                 (time: 84872, pa_bin: 16, energy: 64)
Coordinates:
  * time                    (time) datetime64[ns] 679kB 2016-05-01T00:00:01.9...
  * energy                  (energy) float32 256B 4.628e+03 4.119e+03 ... 3.0
  * pa_bin                  (pa_bin) int64 128B 0 1 2 3 4 5 ... 11 12 13 14 15
Data variables: (12/17)
    def                     (time, pa_bin, energy) float32 348MB dask.array<chunksize=(10000, 16, 64), meta=np.ndarray>
    pa values               (time, pa_bin, energy) float32 348MB dask.array<chunksize=(10000, 16, 64), meta=np.ndarray>
    b_azim                  (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    b_elev                  (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    altitude                (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
    SZA                     (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
  

In [23]:
#The template 
df_8020_template = np.array([
    0.20805, 0.20805, 0.31019, 0.31019, 0.09985, 0.09985, 
    0.12199, 0.12199, 0.16288, 0.16288, 0.04065, 0.04065
])

In [24]:
# Slice the Data to the 20-80 eV range
# MAVEN energy is sorted High -> Low (4000 -> 3). 
# So we slice from 80 down to 20.
ds_subset = df_final.sel(energy=slice(80, 20))

# Safety Check: Ensure the number of bins matches the template
n_bins = ds_subset.sizes['energy']
print(f"Selected Energy Range: {ds_subset.energy.values}")
print(f"Number of bins found: {n_bins}")

if n_bins != 12:
    print("WARNING: The number of energy bins in the 20-80eV range does not match the template (12)!")
    print("Check if your template is meant for a different binning or energy range.")
else:
    # 3. Calculate Shape Parameter (Time Series)
    # Formula: Sum of Absolute Differences over the energy dimension
    
    # For Anti-Parallel
    diff_anti = np.abs(ds_subset['slope_antiparallel'] - df_8020_template)
    shape_a = diff_anti.sum(dim='energy')
    
    # For Parallel
    diff_para = np.abs(ds_subset['slope_parallel'] - df_8020_template)
    shape_p = diff_para.sum(dim='energy')

    # 4. Save back to the main dataset
    df_final['shape_parameter_antiparallel'] = shape_a
    df_final['shape_parameter_parallel'] = shape_p

    print("\nShape parameters calculated successfully!")
    print(df_final)

Selected Energy Range: [78.36285  69.743454 62.07214  55.244617 49.16808  43.759914 38.946617
 34.662746 30.850075 27.456772 24.43671  21.748835]
Number of bins found: 12

Shape parameters calculated successfully!
<xarray.Dataset> Size: 791MB
Dimensions:                       (time: 84872, pa_bin: 16, energy: 64)
Coordinates:
  * time                          (time) datetime64[ns] 679kB 2016-05-01T00:0...
  * energy                        (energy) float32 256B 4.628e+03 ... 3.0
  * pa_bin                        (pa_bin) int64 128B 0 1 2 3 4 ... 12 13 14 15
Data variables: (12/19)
    def                           (time, pa_bin, energy) float32 348MB dask.array<chunksize=(10000, 16, 64), meta=np.ndarray>
    pa values                     (time, pa_bin, energy) float32 348MB dask.array<chunksize=(10000, 16, 64), meta=np.ndarray>
    b_azim                        (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    b_elev                        (time) float32 339kB das

## Create topology column 

In [25]:
ds = df_final     # rename for readability

In [26]:
# 1 — first identify regions
dayside = ds.SZA < 90
nightside = ds.SZA > 110

# 2 — classify electron type in each direction
par_is_sw  = ds.shape_parameter_parallel > 1
apar_is_sw = ds.shape_parameter_antiparallel > 1

par_is_ph  = ds.shape_parameter_parallel < 1
apar_is_ph = ds.shape_parameter_antiparallel < 1

topo = xr.full_like(ds.b_elev, fill_value="NA", dtype=object)

closed_day = dayside & par_is_ph & apar_is_ph

# Open: one PE & one SW
open_day = dayside & (
    (par_is_ph  & apar_is_sw) |
    (par_is_sw  & apar_is_ph)
)

draped_day = dayside & par_is_sw & apar_is_sw

# determine which direction is inward based on b_elev
parallel_inward  = ds.b_elev < 0
anti_inward      = ds.b_elev > 0

closed_night = nightside & (
    (parallel_inward & par_is_ph) |
    (anti_inward     & apar_is_ph)
)

# Open nightside: Ph outward + SW inward
open_night = nightside & (
    (parallel_inward & par_is_sw & apar_is_ph) |
    (anti_inward     & apar_is_sw & par_is_ph)
)

# Draped nightside: SW in both directions
draped_night = nightside & par_is_sw & apar_is_sw

energy_target = 40.0
energy_idx = np.abs(ds.energy - energy_target).argmin()

flux_par_40  = ds.calc_flux_parallel.isel(energy=energy_idx)
flux_apar_40 = ds.calc_flux_antiparallel.isel(energy=energy_idx)

flux_threshold = 1e5  # eV cm^-2 s^-1 sr^-1 eV^-1

void_condition = (flux_par_40 < flux_threshold) & \
                 (flux_apar_40 < flux_threshold)
void_night = nightside & void_condition

topo = topo.where(~closed_day,  "Closed")
topo = topo.where(~open_day,    "Open")
topo = topo.where(~draped_day,  "Draped")

topo = topo.where(~closed_night, "Closed")
topo = topo.where(~open_night,   "Open")
topo = topo.where(~draped_night, "Draped")
topo = topo.where(~void_night, "Void")

In [27]:
ds = ds.assign(topology=topo)

print(ds)


<xarray.Dataset> Size: 792MB
Dimensions:                       (time: 84872, pa_bin: 16, energy: 64)
Coordinates:
  * time                          (time) datetime64[ns] 679kB 2016-05-01T00:0...
  * energy                        (energy) float32 256B 4.628e+03 ... 3.0
  * pa_bin                        (pa_bin) int64 128B 0 1 2 3 4 ... 12 13 14 15
Data variables: (12/20)
    def                           (time, pa_bin, energy) float32 348MB dask.array<chunksize=(10000, 16, 64), meta=np.ndarray>
    pa values                     (time, pa_bin, energy) float32 348MB dask.array<chunksize=(10000, 16, 64), meta=np.ndarray>
    b_azim                        (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    b_elev                        (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    altitude                      (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
    SZA                           (time) float64 679kB dask.array<

In [28]:
import gc

# 1. Create your new smaller dataset
vars_to_drop = [v for v in ds.data_vars if ds[v].dims != ('time',)]
ds_timeonly = ds.drop_vars(vars_to_drop)

# 2. Close and Delete the old 'ds'
ds.close() # Closes the actual file link
del ds     # Deletes the variable name
gc.collect() # Forces Python to clean up the garbage memory


139

In [29]:
print(ds_timeonly)

<xarray.Dataset> Size: 10MB
Dimensions:                       (time: 84872, energy: 64, pa_bin: 16)
Coordinates:
  * time                          (time) datetime64[ns] 679kB 2016-05-01T00:0...
  * energy                        (energy) float32 256B 4.628e+03 ... 3.0
  * pa_bin                        (pa_bin) int64 128B 0 1 2 3 4 ... 12 13 14 15
Data variables: (12/14)
    b_azim                        (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    b_elev                        (time) float32 339kB dask.array<chunksize=(10000,), meta=np.ndarray>
    altitude                      (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
    SZA                           (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
    B_x                           (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
    B_y                           (time) float64 679kB dask.array<chunksize=(10000,), meta=np.ndarray>
    ...   

In [30]:
# 1. Drop the dimensions you don't need (same as before)
df_1d_summary = ds_timeonly.drop_dims(['energy', 'pa_bin'])

# 2. Convert to a Dask DataFrame (Lazy)
# This creates a "blueprint" for the table without loading it into RAM yet
ddf = df_1d_summary.to_pandas()

# 3. Save directly to CSV using Dask (Efficient)
# This will compute and write the file piece-by-piece
output_filename = '/Users/quocanh1306/Downloads/topo_maven_05_2016.csv'
ddf.to_csv(output_filename, index=True)

print("Saved successfully! Check your downloads folder.")

Saved successfully! Check your downloads folder.
